<a href="https://colab.research.google.com/github/jhenningsen/Equity_Analysis/blob/main/LangStudio/RSI_Recent_Detail_Summary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import yfinance as yf
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

### Remember to load the OptionVolume.csv file before running the scripts

In [63]:
# Profitability and Summary functions

def calculate_forward_returns(price_df, windows=[3, 5, 10]):
    """
    Orchestrates vectorized return calculations across the standard trio of windows.
    Enforces standard 3, 5, and 10-day windows.[cite: 5]
    """
    df = price_df.copy()
    for w in windows:
        # Vectorized calculation: (Price in W days / Current Price) - 1
        df[f'Ret_{w}d'] = df['Adj Close'].shift(-w) / df['Adj Close'] - 1
    return df

def get_signal_performance(price_df, signals_df, strategy_name="Strategy"):
    """
    Specialized join to isolate forward returns strictly at signal timestamps.
    Eliminates repeated return calculations by checking for existing columns.
    """
    if signals_df.empty:
        return pd.DataFrame()

    # --- ARCHITECT UPDATE: 5-TRADING-DAY LOCKOUT ---
    # We use the integer position (iloc) of the price_df index to measure
    # distance in TRADING DAYS, not calendar days.
    signals_df = signals_df.sort_index()
    trading_day_positions = price_df.index.get_indexer(signals_df.index)

    # Calculate the difference in trading days between signals
    # Use a diff on the integer positions
    days_since_last_trade = pd.Series(trading_day_positions).diff()

    # Apply Lockout Filter
    # Note: .diff() results in NaN for the first entry, which we always keep.
    mask = (days_since_last_trade.isna()) | (days_since_last_trade > 5)
    signals_df = signals_df[mask.values]
    # ----------------------------------------------

    required_cols = ['Ret_3d', 'Ret_5d', 'Ret_10d']
    # Efficiently add return windows only if they don't already exist in price_df
    if not all(col in price_df.columns for col in required_cols):
        price_df = calculate_forward_returns(price_df)

    # Inner join maps price performance to the specific signal dates
    performance_df = signals_df.merge(
        price_df[['Adj Close'] + required_cols],
        left_index=True,
        right_index=True,
        how='inner'
    )

    performance_df['Strategy_Name'] = strategy_name
    return performance_df

def calculate_exposure_metrics(price_df, signals_df, trade_size=10000):
    """
    Calculates Max and Avg Value Risked for 3D, 5D, and 10D windows.
    Strictly vectorized to handle high-frequency ticker scans.
    """
    exposure_results = {}
    windows = [3, 5, 10]

    # Ensure index is datetime for proper alignment
    full_time_index = price_df.index

    for days in windows:
        # Create a zero-series based on the full price history
        exposure_series = pd.Series(0, index=full_time_index)

        # Mark each signal entry
        # We assume signals_df index aligns with price_df index
        signal_dates = signals_df.index

        # Vectorized pulse: For each signal, add trade_size for the next 'days' trading days
        # We use a shift-sum approach to avoid nested loops
        daily_entries = pd.Series(0, index=full_time_index)
        daily_entries.loc[signal_dates] = trade_size

        # Calculate rolling sum of entries to find active positions
        # A 3-day window means a trade is active on Day 0, 1, and 2.
        active_exposure = daily_entries.rolling(window=days, min_periods=1).sum()

        exposure_results[f'Max_Risked_{days}D'] = active_exposure.max()
        exposure_results[f'Avg_Risked_{days}D'] = active_exposure.mean()

    return exposure_results


def display_strategy_summary(perf_df, trade_size=10000):
    """
    Architectural Grade Summary: Restores Alpha Drivers and injects Risk Metrics.
    Standardizes on 3, 5, and 10-day forward return windows.
    """
    if perf_df.empty:
        print("⚠️ ARCHITECT NOTICE: No signals to analyze.")
        return

    # 1. DATA VALIDATION: Ensure 10-Day exists (Universal Standard)
    # If Day_10_$ is missing in your current fetch, this will alert you.
    windows = [1, 2, 3, 4, 5, 10]
    available_days = [d for d in windows if f'Day_{d}_$' in perf_df.columns]

    # 2. STATISTICAL RIGOR CHECK
    total_signals = len(perf_df)
    print("\n" + "═" * 100)
    print(f"🚀 SENIOR ARCHITECT: ENHANCED PERFORMANCE & RISK AUDIT")
    if total_signals < 30:
        print(f"❌ STATISTICAL ALERT: Trade_Count ({total_signals}) < 30 threshold.")
        print("   Variance likely dominated by noise. Results should be treated as anecdotal.")
    print("-" * 100)

    # 3. VECTORIZED METRICS (Including Avg/Max at Risk)
    summary_list = []

    for d in available_days:
        col = f'Day_{d}_$'

        # Risk Calculation: In a simple pnl-table, we look at the min value
        # across all columns up to day 'd' to find the Max Drawdown (Risk) per trade.
        cols_up_to_d = [f'Day_{i}_$' for i in available_days if i <= d]
        max_drawdowns = perf_df[cols_up_to_d].min(axis=1)
        # Only count negative values as 'at risk'
        at_risk_per_trade = max_drawdowns.apply(lambda x: x if x < 0 else 0)

        summary_list.append({
            "Window": f"Day {d}",
            "Total Profit": f"${perf_df[col].sum():,.2f}",
            "Win Rate": f"{(perf_df[col] > 0).mean():.1%}",
            "Avg/Trade": f"${perf_df[col].mean():,.2f}",
            "Avg At Risk": f"${at_risk_per_trade.mean():,.2f}",
            "Max At Risk": f"${at_risk_per_trade.min():,.2f}",
            "Std Dev": f"${perf_df[col].std():,.2f}"
        })

    print(pd.DataFrame(summary_list).to_string(index=False))

    # 4. RESTORING TOP 5 ALPHA DRIVERS (Based on max available window)
    if available_days:
        best_window_col = f'Day_{max(available_days)}_$'
        print(f"\n🏆 TOP 5 ALPHA DRIVERS ({max(available_days)}-Day Cumulative):")
        top_5 = perf_df.sort_values(by=best_window_col, ascending=False).head(5)
        for _, row in top_5.iterrows():
            print(f" • {row['Symbol']}: ${row[best_window_col]:,.2f}")

    # 5. CRITIQUE ENGINE
    if 'Day_10_$' not in perf_df.columns:
        print("\n⚠️ COMPLIANCE WARNING: 10-Day Alpha window missing. Update fetch logic.")

    print("═" * 100)

In [58]:
# ==========================================
# ⚙️ GLOBAL CONFIGURATION (TUNING)
# ==========================================
# Format: (Length, Threshold)
RSI_CONFIG = [
    (16, 25),
    (24, 30),
    (14, 25),
    (22, 30),
    (18, 25),
    (22, 25),
    (26, 30)
]
LOOKBACK_DAYS = 30      # How far back to search for trigger events
CSV_FILE = "OptionVolume.csv"  # Source for your ticker universe
FALLBACK_SYMBOLS = ["TSLA", "NVDA", "AAPL", "AMD", "MSFT", "BTC-USD"]

# ==========================================
# 📈 PRECISION CALCULATION LOGIC
# ==========================================

def calculate_rsi_precision(series, period=14):
    """Matches Yahoo Finance / Wilder's Smoothing Precision"""
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    # com=period-1 is the mathematical equivalent to Wilder's Smoothing
    avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()

    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

def scan_for_triggers():
    """Main scanning engine for recent RSI events"""
    # 1. Load Symbols
    try:
        df_csv = pd.read_csv(CSV_FILE)
        symbol_col = [c for c in df_csv.columns if 'symbol' in c.lower()][0]
        symbols = df_csv[symbol_col].str.strip().unique().tolist()
        print(f"✅ Loaded {len(symbols)} symbols from {CSV_FILE}")
    except:
        symbols = FALLBACK_SYMBOLS
        print(f"⚠️ {CSV_FILE} not found. Using fallback list: {symbols}")

    report_data = []

    # Calculate dates
    # Fetch 300 extra days to allow RSI math to stabilize (Wilder's Warm-up)
    start_fetch = (datetime.now() - timedelta(days=LOOKBACK_DAYS + 300)).strftime('%Y-%m-%d')
    trigger_cutoff = (datetime.now() - timedelta(days=LOOKBACK_DAYS)).date()

    print(f"🔍 Scanning for RSI triggers using {len(RSI_CONFIG)} configurations since {trigger_cutoff}...")

    for symbol in symbols:
        try:
            # Fetch data once per symbol
            df = yf.download(symbol, start=start_fetch, progress=False, auto_adjust=True)
            if df.empty: continue

            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            # NEW: Loop through each Length/Threshold pair
            for length, threshold in RSI_CONFIG:
                # Calculate RSI for this specific length
                rsi_col = f'RSI_{length}'
                df[rsi_col] = calculate_rsi_precision(df['Close'], period=length)

                # TRIGGER: Check against the specific threshold for this length
                trigger_col = f'Trigger_{length}'
                df[trigger_col] = (df[rsi_col] < threshold) & (df[rsi_col].shift(1) >= threshold)

                # Filter for the lookback window
                recent_hits = df[df.index.date >= trigger_cutoff]
                recent_hits = recent_hits[recent_hits[trigger_col] == True]

                for date, row in recent_hits.iterrows():
                    report_data.append({
                        "Date": date.strftime('%Y-%m-%d'),
                        "Symbol": symbol,
                        "Price": round(row['Close'], 2),
                        "RSI_Len": length,           # Added to track which pair triggered
                        "RSI_Val": round(row[rsi_col], 2),
                        "Thresh": threshold          # Added for clarity
                    })
        except Exception as e:
            print(f"❌ Error processing {symbol}: {e}")
            continue

    return report_data

# ==========================================
# 📝 EXECUTION & OUTPUT GENERATION
# ==========================================

if __name__ == "__main__":
    triggers = scan_for_triggers()

    print("\n" + "="*70)
    print(f"🚀 RSI MULTI-PAIR REPORT: LAST {LOOKBACK_DAYS} DAYS")
    print(f"Pairs Scanned: {RSI_CONFIG}")
    print("="*70)

    if not triggers:
        print(f"No triggers found in the last {LOOKBACK_DAYS} days.")
    else:
        results_df = pd.DataFrame(triggers)
        # Sort by Date (newest) then Symbol
        results_df = results_df.sort_values(by=["Date", "Symbol"], ascending=[False, True])
        print(results_df.to_string(index=False))

    print("="*70)

✅ Loaded 200 symbols from OptionVolume.csv
🔍 Scanning for RSI triggers using 7 configurations since 2026-04-04...

🚀 RSI MULTI-PAIR REPORT: LAST 30 DAYS
Pairs Scanned: [(16, 25), (24, 30), (14, 25), (22, 30), (18, 25), (22, 25), (26, 30)]
      Date Symbol  Price  RSI_Len  RSI_Val  Thresh
2026-05-04    ABT  87.54       24    29.08      30
2026-05-04    ABT  87.54       22    28.53      30
2026-05-04    ABT  87.54       26    29.59      30
2026-05-01    LMT 512.77       16    24.84      25
2026-05-01    LMT 512.77       22    29.86      30
2026-04-24    ABT  91.13       24    29.60      30
2026-04-24    ABT  91.13       22    29.00      30
2026-04-24    LMT 513.45       24    29.88      30
2026-04-24    LMT 513.45       22    28.16      30
2026-04-24    LMT 513.45       18    24.13      25
2026-04-24    RTX 174.26       14    24.27      25
2026-04-23    LMT 529.79       16    24.16      25
2026-04-23    LMT 529.79       14    21.53      25
2026-04-22    ABT  91.70       24    29.18     

In [64]:
# --- Execution ---
if 'results_df' in locals() and not results_df.empty:
    profit_df = calculate_signal_profitability(results_df)

    # Display Summary
    cols_to_show = ['Date', 'Symbol', 'Price', 'Trend'] + [f'Day_{i}_$' for i in range(1, 6)]
    # Filter columns that exist (Trend is only in the Momentum Ignition results)
    actual_cols = [c for c in cols_to_show if c in profit_df.columns]

    print("\n" + "💸" * 20)
    print(f"PROFITABILITY REPORT (Max ${10000:,.0f} per Trade)")
    print("💸" * 20)
    print(profit_df[actual_cols].to_string(index=False))
else:
    print("⚠️ No signals found in results_df to analyze.")

# --- Execution ---
display_strategy_summary(profit_df)

📊 Analyzing profitability for 15 unique signals...

💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸
PROFITABILITY REPORT (Max $10,000 per Trade)
💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸
      Date Symbol  Price  Day_1_$  Day_2_$  Day_3_$  Day_4_$  Day_5_$
2026-05-04    ABT  87.54      NaN      NaN      NaN      NaN      NaN
2026-05-01    LMT 512.77   104.92      NaN      NaN      NaN      NaN
2026-04-24    ABT  91.13   183.26   299.57    21.95   -37.31  -183.25
2026-04-24    LMT 513.45    -1.95   -22.59   -70.89    88.03   -13.24
2026-04-24    RTX 174.26   -50.50    81.49   -84.36   103.87   -15.49
2026-04-23    LMT 529.79  -308.42  -310.31  -330.32  -377.13  -223.11
2026-04-22    ABT  91.70    85.06   -62.16   119.96   235.55   -40.35
2026-04-21    ABT  92.72  -110.01   -25.88  -171.48     8.63   122.95
2026-04-10    NKE  42.62    68.04   370.72   661.66   722.67   800.09
2026-04-10    NOW  83.00   730.12   577.11  1348.19  1619.28  1645.78
2026-04-10   SNOW 121.11  1084.14  1185.70  1929.65  1852.86  1888.37
2026-04-10   TEAM  57

In [36]:
# ==========================================
# ⚙️ GLOBAL CONFIGURATION (MOMENTUM TUNE)
# ==========================================
# Format: (Length, Threshold)
RSI_CONFIG = [
    (10, 80),
    (14, 75)
]
LOOKBACK_DAYS = 30      # Lookback window for trigger events
SMA_TREND = 200         # Baseline trend filter
CSV_FILE = "OptionVolume.csv"
FALLBACK_SYMBOLS = ["TSLA", "NVDA", "AAPL", "AMD", "MSFT", "COIN", "MSTR"]

# ==========================================
# 📈 PRECISION CALCULATION LOGIC
# ==========================================

def calculate_rsi_precision(series, period=14):
    """Matches Wilder's Smoothing Precision (Yahoo Style)"""
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    # com=period-1 is equivalent to Wilder's alpha=1/period
    avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()

    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

def scan_for_momentum_triggers():
    """Identifies 'Momentum Ignition' (RSI crossing ABOVE threshold)"""
    try:
        df_csv = pd.read_csv(CSV_FILE)
        symbol_col = [c for c in df_csv.columns if 'symbol' in c.lower()][0]
        symbols = df_csv[symbol_col].str.strip().unique().tolist()
    except:
        symbols = FALLBACK_SYMBOLS

    report_data = []

    # Fetch 350 days to ensure SMA 200 and RSI 10 are both stable
    start_fetch = (datetime.now() - timedelta(days=LOOKBACK_DAYS + 350)).strftime('%Y-%m-%d')
    trigger_cutoff = (datetime.now() - timedelta(days=LOOKBACK_DAYS)).date()

    print(f"🚀 Scanning for Momentum Ignition (RSI configurations: {RSI_CONFIG}) since {trigger_cutoff}...")

    for symbol in symbols:
        try:
            df = yf.download(symbol, start=start_fetch, progress=False, auto_adjust=True)
            if df.empty or len(df) < SMA_TREND: continue

            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            df['SMA_200'] = df['Close'].rolling(window=SMA_TREND).mean()
            df['Trend'] = np.where(df['Close'] > df['SMA_200'], "UP", "DOWN")

            # NEW: Evaluate each specific RSI pair
            for length, threshold in RSI_CONFIG:
                rsi_vals = calculate_rsi_precision(df['Close'], period=length)

                # TRIGGER: Today is above threshold, Yesterday was below (Cross Above)
                trigger = (rsi_vals > threshold) & (rsi_vals.shift(1) <= threshold)

                # Filter for the lookback window
                recent_hits = df[df.index.date >= trigger_cutoff].copy()
                recent_hits['Is_Trigger'] = trigger
                hits = recent_hits[recent_hits['Is_Trigger'] == True]

                for date, row in hits.iterrows():
                    report_data.append({
                        "Date": date.strftime('%Y-%m-%d'),
                        "Symbol": symbol,
                        "Price": f"${row['Close']:.2f}",
                        "RSI_Len": length,           # Track which length triggered
                        "RSI_Val": round(rsi_vals.loc[date], 1),
                        "Thresh": threshold,         # Track the target threshold
                        "Trend": row['Trend']
                    })
        except Exception:
            continue

    return report_data

# ==========================================
# 📝 EXECUTION & OUTPUT GENERATION
# ==========================================

if __name__ == "__main__":
    triggers = scan_for_momentum_triggers()

    print("\n" + "="*80)
    print(f"🚀 RSI MOMENTUM POWER ZONE REPORT (LAST {LOOKBACK_DAYS} DAYS)")
    print(f"Pairs Scanned: {RSI_CONFIG}")
    print("="*80)

    if not triggers:
        print(f"No momentum breakouts detected in the last {LOOKBACK_DAYS} days.")
    else:
        results_df = pd.DataFrame(triggers)
        # Sort by Date (newest first)
        results_df = results_df.sort_values(by="Date", ascending=False)

        print(results_df.to_string(index=False))

    print("="*80)


🚀 Scanning for Momentum Ignition (RSI configurations: [(10, 80), (14, 75)]) since 2026-04-04...

🚀 RSI MOMENTUM POWER ZONE REPORT (LAST 30 DAYS)
Pairs Scanned: [(10, 80), (14, 75)]
      Date Symbol    Price  RSI_Len  RSI_Val  Thresh Trend
2026-05-04   SNDK $1255.86       14     76.5      75    UP
2026-05-04     MU  $576.45       10     80.5      80    UP
2026-05-04     MU  $576.45       14     76.2      75    UP
2026-05-04    WDC  $442.36       10     81.3      80    UP
2026-05-01     ZM  $103.44       14     77.0      75    UP
2026-05-01     BE  $290.52       10     80.8      80    UP
2026-05-01   TWLO  $183.34       10     83.6      80    UP
2026-05-01   AMZN  $268.26       10     81.3      80    UP
2026-05-01   TWLO  $183.34       14     79.8      75    UP
2026-05-01     ZM  $103.44       10     81.4      80    UP
2026-04-30   MRVL  $165.15       14     76.9      75    UP
2026-04-30  GOOGL  $384.80       10     87.1      80    UP
2026-04-30    AMD  $354.49       10     80.1      80

In [37]:
# --- Execution ---
if 'results_df' in locals() and not results_df.empty:
    profit_df = calculate_signal_profitability(results_df)

    # Display Summary
    cols_to_show = ['Date', 'Symbol', 'Price', 'Trend'] + [f'Day_{i}_$' for i in range(1, 6)]
    # Filter columns that exist (Trend is only in the Momentum Ignition results)
    actual_cols = [c for c in cols_to_show if c in profit_df.columns]

    print("\n" + "💸" * 20)
    print(f"PROFITABILITY REPORT (Max ${10000:,.0f} per Trade)")
    print("💸" * 20)
    print(profit_df[actual_cols].to_string(index=False))
else:
    print("⚠️ No signals found in results_df to analyze.")

# --- Execution ---
display_strategy_summary_v2(profit_df)


📊 Analyzing profitability for 86 unique signals...

💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸
PROFITABILITY REPORT (Max $10,000 per Trade)
💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸💸
      Date Symbol    Price Trend  Day_1_$  Day_2_$  Day_3_$  Day_4_$  Day_5_$
2026-05-04   SNDK $1255.86    UP      NaN      NaN      NaN      NaN      NaN
2026-05-04     MU  $576.45    UP      NaN      NaN      NaN      NaN      NaN
2026-05-04    WDC  $442.36    UP      NaN      NaN      NaN      NaN      NaN
2026-05-01     ZM  $103.44    UP   308.39      NaN      NaN      NaN      NaN
2026-05-01     BE  $290.52    UP   -64.71      NaN      NaN      NaN      NaN
2026-05-01   TWLO  $183.34    UP   345.26      NaN      NaN      NaN      NaN
2026-05-01   AMZN  $268.26    UP   141.28      NaN      NaN      NaN      NaN
2026-04-30   MRVL  $165.15    UP   -12.11   -90.22      NaN      NaN      NaN
2026-04-30  GOOGL  $384.80    UP    23.13   -40.28      NaN      NaN      NaN
2026-04-30    AMD  $354.49    UP   170.67  -365.31      NaN      NaN      NaN
202